# 05 - Causal Inference When You Cannot Randomise
**Milestone 6**

Three observational designs, each with its own identification story:

1. **Propensity-score matching** (confounded cross-section, + DoWhy)
2. **Difference-in-differences** (Card & Krueger minimum wage)
3. **Synthetic control** (California Proposition 99)


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Propensity-score matching on confounded data
The naive difference-in-means is biased; matching on the propensity score removes most of the confounding (true effect = **+5**).

In [ ]:
from src.data_pipeline.simulate import simulate_confounded_observational
from src.core.causal import propensity as psm

obs = simulate_confounded_observational(n=5000, seed=5)
out = psm.propensity_score_matching(
    obs, treatment="treatment", outcome="outcome",
    covariates=["age", "prior_spend"], caliper=0.2, n_bootstrap=300,
)
{ "naive_estimate": round(out["naive_estimate"], 3),
  "psm_estimate": round(out["effect"], 3),
  "true_effect": 5.0,
  "ci": [round(v, 3) for v in out["ci"]],
  "max_smd_before": round(out["max_smd_before"], 3),
  "max_smd_after": round(out["max_smd_after"], 3),
  "balanced": out["balanced"] }

### Covariate balance before vs. after matching
Standardised mean differences should fall below the 0.1 rule of thumb.

In [ ]:
cov = list(out["balance_smd_before"].keys())
before = [out["balance_smd_before"][k] for k in cov]
after = [out["balance_smd_after"][k] for k in cov]
x = np.arange(len(cov))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, before, 0.4, label="before matching", color="#f87171")
ax.bar(x + 0.2, after, 0.4, label="after matching", color="#34d399")
ax.axhline(0.1, ls="--", color="#4f46e5", label="0.1 threshold")
ax.set_xticks(x); ax.set_xticklabels(cov); ax.set_ylabel("standardised mean diff")
ax.set_title("Covariate balance"); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "05_balance.png"), dpi=120, bbox_inches="tight"); plt.show()

### Cross-check with DoWhy (`backdoor.propensity_score_matching`) plus a refutation test

In [ ]:
dowhy_out = psm.dowhy_propensity_estimate(
    obs, treatment="treatment", outcome="outcome",
    covariates=["age", "prior_spend"], refute=True,
)
dowhy_out

## 2. Difference-in-differences - Card & Krueger minimum wage
NJ raised its minimum wage in 1991; PA did not. The interaction term is the ATT.

In [ ]:
from src.data_pipeline import datasets
from src.core.causal import did

ck = datasets.load_card_krueger()
did_res = did.difference_in_differences(ck, outcome="fte_employment", treated_col="treated", post_col="post")
{ "att": round(did_res["att"], 3), "ci": [round(v, 3) for v in did_res["ci"]],
  "p_value": f"{did_res['p_value']:.2e}", "r_squared": round(did_res["r_squared"], 3) }

In [ ]:
cm = did_res["cell_means"]
fig, ax = plt.subplots(figsize=(7, 4))
periods = ["pre", "post"]
ax.plot(periods, [cm["control_pre"], cm["control_post"]], "o-", label="control (PA)", color="#94a3b8")
ax.plot(periods, [cm["treated_pre"], cm["treated_post"]], "o-", label="treated (NJ)", color="#4f46e5")
ax.set_ylabel("FTE employment"); ax.set_title(f"DiD ATT = {did_res['att']:.2f}")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "05_did.png"), dpi=120, bbox_inches="tight"); plt.show()

## 3. Synthetic control - California Proposition 99
We build a weighted 'synthetic California' from donor states and read the post-1988 gap as the effect.

In [ ]:
from src.core.causal import synthetic_control as sc

prop = datasets.load_prop99()
sc_res = sc.synthetic_control(prop, unit_col="state", time_col="year",
                              outcome_col="cigsales", treated_unit="California",
                              treatment_time=1989)
{ "effect": round(sc_res["effect"], 3), "pre_period_rmse": round(sc_res["pre_period_rmse"], 3),
  "p_value": round(sc_res["p_value"], 3), "n_donors": sc_res["n_donors"] }

In [ ]:
years = [int(y) for y in sc_res["time"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(years, sc_res["treated"], "-", label="California (actual)", color="#4f46e5")
ax.plot(years, sc_res["synthetic"], "--", label="Synthetic California", color="#f59e0b")
ax.axvline(1989, color="#dc2626", ls=":", label="Proposition 99 (1989)")
ax.set_xlabel("year"); ax.set_ylabel("per-capita cigarette sales")
ax.set_title("Synthetic control: California vs. synthetic counterfactual")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "05_synthetic_control.png"), dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Top donors in the synthetic control
pd.DataFrame(sc_res["weights"]).head(8)

## Takeaways
* Observational estimates are only as good as their identification assumptions - always check balance / pre-trends / pre-period fit.
* PSM removes measured confounding; DiD removes time-invariant unit confounders; synthetic control builds an explicit counterfactual from untreated units.
* DoWhy's refutation tests and the synthetic-control placebo test quantify how fragile each estimate is.